without cleaning , (duplicates) can lead to bias

In [5]:
import sqlite3
import pandas as pd
import os

In [36]:
db_path = os.path.join('database.sqlite')
connection = sqlite3.connect(db_path)

In [80]:
# loading dataset
df = pd.read_sql_query(  # not considering score = 3 (to avoid neutral ambiguity , only +ve and -ve)
"""
select *
from reviews
where score != 3 
order by ProductId
""", connection)
df.head(2)
df.tail(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
525812,5703,B009WSNWC4,AMP7K1O84DH1T,ESTY,0,0,5,1351209600,DELICIOUS,Purchased this product at a local store in NY ...
525813,327601,B009WVB40S,A3ME78KVX31T21,K'la,0,0,5,1351123200,Tasty!,I purchased this to send to my son who's away ...


In [81]:
def partition(x):  # [1,2] => -ve , [4,5] => +ve
    if x < 3:    
        return 'negative'
    return 'positive'

score = df['Score']
positive_negative = score.map(partition)
df['Score'] = positive_negative


In [82]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,150493,0006641040,AMX0PJKV4PPNJ,"E. R. Bird ""Ramseelbird""",71,72,positive,1096416000,Read it once. Read it twice. Reading Chicken S...,"These days, when a person says, ""chicken soup""..."
1,150494,0006641040,AYZ0PR5QZROD1,Mother of 3 girls,3,3,positive,1173312000,Family favorite,All of my children love this book. My first g...
2,150496,0006641040,A3KKR87BJ0C595,"Gretchen Goodfellow ""Lover of children's lit""",3,3,positive,1111363200,"You'll use it once, you'll use it twice",One of my earliest memories is of this book. ...
3,150497,0006641040,A1HKYQOFC8ZZCH,"Maria Apolloni ""lanarossa""",2,2,negative,1334707200,"The story is great, the softcover book is disa...",I give five stars to the Maurice Sendak story....
4,150498,0006641040,A3SJWISOCP31TR,R. J. Wells,2,2,positive,1176336000,A Gem of a Book,This is a wonderful little book. I loved it 40...


In [83]:
sorted_data = df.sort_values('ProductId', axis = 0, ascending=True, inplace=False)
sorted_data

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,150493,0006641040,AMX0PJKV4PPNJ,"E. R. Bird ""Ramseelbird""",71,72,positive,1096416000,Read it once. Read it twice. Reading Chicken S...,"These days, when a person says, ""chicken soup""..."
19,150513,0006641040,ASH0DZQQF6AIZ,tessarat,0,0,positive,1325721600,A classic,I remembered this book from my childhood and g...
20,150514,0006641040,A2ONB6ZA292PA,Rosalind Matzner,0,0,positive,1313884800,Chicken soup with Rice,It's a great book with adorable illustrations....
21,150515,0006641040,A2RTT81R6Y3R7X,Lindylu,0,0,positive,1303171200,One of our family's favorite books,This book is a family favorite and was read to...
22,150516,0006641040,A3OI7ZGH6WZJ5G,"Mary Jane Rogers ""Maedchen""",0,0,positive,1293840000,Darling!,"The same author wrote ""Where the Wild Things A..."
...,...,...,...,...,...,...,...,...,...,...
525809,191721,B009UOFTUI,AJVB004EB0MVK,D. Christofferson,0,0,negative,1345852800,weak coffee not good for a premium product and...,"This coffee supposedly is premium, it tastes w..."
525810,1478,B009UOFU20,AJVB004EB0MVK,D. Christofferson,0,0,negative,1345852800,weak coffee not good for a premium product and...,"This coffee supposedly is premium, it tastes w..."
525811,328482,B009UUS05I,ARL20DSHGVM1Y,Jamie,0,0,positive,1331856000,Perfect,The basket was the perfect sympathy gift when ...
525812,5703,B009WSNWC4,AMP7K1O84DH1T,ESTY,0,0,positive,1351209600,DELICIOUS,Purchased this product at a local store in NY ...


In [84]:
df2  =  sorted_data.drop_duplicates(subset={'UserId', 'ProfileName', 'Time', 'Summary', 'Text'})
df2

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,150493,0006641040,AMX0PJKV4PPNJ,"E. R. Bird ""Ramseelbird""",71,72,positive,1096416000,Read it once. Read it twice. Reading Chicken S...,"These days, when a person says, ""chicken soup""..."
19,150513,0006641040,ASH0DZQQF6AIZ,tessarat,0,0,positive,1325721600,A classic,I remembered this book from my childhood and g...
20,150514,0006641040,A2ONB6ZA292PA,Rosalind Matzner,0,0,positive,1313884800,Chicken soup with Rice,It's a great book with adorable illustrations....
21,150515,0006641040,A2RTT81R6Y3R7X,Lindylu,0,0,positive,1303171200,One of our family's favorite books,This book is a family favorite and was read to...
22,150516,0006641040,A3OI7ZGH6WZJ5G,"Mary Jane Rogers ""Maedchen""",0,0,positive,1293840000,Darling!,"The same author wrote ""Where the Wild Things A..."
...,...,...,...,...,...,...,...,...,...,...
525782,193169,B009RSR8HO,A3M3S2NCVZ8UXF,Stephanie,0,0,positive,1350518400,Fat to Skinny zero,I have been using this product for 4 months no...
525806,188389,B009SF0TN6,A1L0GWGRK4BYPT,Bety Robinson,0,0,positive,1350518400,Amazing!! Great sauce for everything!,You have to try this sauce to believe it! It s...
525808,221795,B009SR4OQ2,A32A6X5KCP7ARG,sicamar,1,1,positive,1350604800,Awesome Taste,I bought this Hazelnut Paste (Nocciola Spread)...
525812,5703,B009WSNWC4,AMP7K1O84DH1T,ESTY,0,0,positive,1351209600,DELICIOUS,Purchased this product at a local store in NY ...


In [85]:
(df2['Id'].size * 1.0) / (df['Id'].size * 1.0) * 100

69.4795117665182

In [86]:
display = pd.read_sql_query(
"""
SELECT *
FROM Reviews
WHERE Score != 3 AND Id= 544173 or Id = 64422
ORDER BY ProductId
""", connection)
display

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,64422,B000MIDROQ,A161DK06JJMCYF,"J. E. Stephens ""Jeanne""",3,1,5,1224892800,Bought This for My Son at College,My son loves spaghetti so I didn't hesitate or...
1,544173,B000U9WZ54,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...


In [87]:
final = df2[df2.HelpfulnessNumerator <= df2.HelpfulnessDenominator]


In [ ]:
print(final.shape)

final['Score'].value_counts()


(365331, 10)


Score
positive    307967
negative     57364
Name: count, dtype: int64

In [92]:
final.duplicated().sum()

np.int64(0)

In [93]:
final.isna().sum()

Id                        0
ProductId                 0
UserId                    0
ProfileName               0
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   0
Text                      0
dtype: int64

In [97]:
final.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Time
count,365331.000000,365331.000000,365331.000000,3.653310e+05
mean,282791.666398,1.739182,2.188040,1.296127e+09
std,164586.488829,6.721752,7.347319,4.865088e+07
min,1.000000,0.000000,0.000000,9.393408e+08
25%,140723.500000,0.000000,0.000000,1.270858e+09
50%,278958.000000,0.000000,1.000000,1.311379e+09
75%,428530.500000,2.000000,2.000000,1.332893e+09
max,568454.000000,866.000000,878.000000,1.351210e+09


In [ ]:
#convert to csv 
final.to_csv("./reviews_clean_file")

In [101]:
df = pd.read_csv("./reviews_clean_file", index_col=0)

In [ ]:
df.info

In [103]:
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,150493,0006641040,AMX0PJKV4PPNJ,"E. R. Bird ""Ramseelbird""",71,72,positive,1096416000,Read it once. Read it twice. Reading Chicken S...,"These days, when a person says, ""chicken soup""..."
19,150513,0006641040,ASH0DZQQF6AIZ,tessarat,0,0,positive,1325721600,A classic,I remembered this book from my childhood and g...
20,150514,0006641040,A2ONB6ZA292PA,Rosalind Matzner,0,0,positive,1313884800,Chicken soup with Rice,It's a great book with adorable illustrations....
21,150515,0006641040,A2RTT81R6Y3R7X,Lindylu,0,0,positive,1303171200,One of our family's favorite books,This book is a family favorite and was read to...
22,150516,0006641040,A3OI7ZGH6WZJ5G,"Mary Jane Rogers ""Maedchen""",0,0,positive,1293840000,Darling!,"The same author wrote ""Where the Wild Things A..."
...,...,...,...,...,...,...,...,...,...,...
525782,193169,B009RSR8HO,A3M3S2NCVZ8UXF,Stephanie,0,0,positive,1350518400,Fat to Skinny zero,I have been using this product for 4 months no...
525806,188389,B009SF0TN6,A1L0GWGRK4BYPT,Bety Robinson,0,0,positive,1350518400,Amazing!! Great sauce for everything!,You have to try this sauce to believe it! It s...
525808,221795,B009SR4OQ2,A32A6X5KCP7ARG,sicamar,1,1,positive,1350604800,Awesome Taste,I bought this Hazelnut Paste (Nocciola Spread)...
525812,5703,B009WSNWC4,AMP7K1O84DH1T,ESTY,0,0,positive,1351209600,DELICIOUS,Purchased this product at a local store in NY ...
